# Plant Disease Detection using Deep Learning

This project presents an end-to-end deep learning pipeline for plant leaf disease classification using CNN-LSTM
and VGG16-LSTM architectures. The workflow includes dataset preprocessing, synthetic image generation using Stable
Diffusion, transfer learning, fine-tuning, and Explainable AI techniques such as Saliency Maps and LIME.

In [ ]:
## Import Libraries

In [ ]:
import zipfile
import os
import random
import shutil
import cv2
import matplotlib.pyplot as plt
import numpy as np
from numpy import prod
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, InputLayer, Reshape, LSTM, TimeDistributed, GlobalAveragePooling2D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow as tf
from tensorflow.keras import backend as K

## Dataset Preparation

The PlantVillage dataset is organized into training and testing directories using an 80:20 split. 
Images are copied into class-specific folders while preserving the original dataset.

In [4]:
dataset_path = "/kaggle/input/plantleaf-unaugmented"

def organize_dataset(base_dir, output_dir="/kaggle/working/organized_dataset"):
    train_dir = os.path.join(output_dir, 'train')
    test_dir = os.path.join(output_dir, 'test')

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for class_name in os.listdir(base_dir):
        class_path = os.path.join(base_dir, class_name)
        if os.path.isdir(class_path) and class_name not in ['train', 'test']:
            os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
            os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

            images = os.listdir(class_path)
            random.shuffle(images)

            split_index = int(0.8 * len(images))
            train_images = images[:split_index]
            test_images = images[split_index:]

            for img in train_images:
                shutil.copy(os.path.join(class_path, img), os.path.join(train_dir, class_name, img))  # ✅ Use copy instead of move
            for img in test_images:
                shutil.copy(os.path.join(class_path, img), os.path.join(test_dir, class_name, img))  # ✅ Use copy instead of move

    return train_dir, test_dir

# ✅ Update dataset path
dataset_path = "/kaggle/input/plantleaf-unaugmented"

# ✅ Organize dataset
train_dir, test_dir = organize_dataset(dataset_path)

In [ ]:
def count_images(directory):
    image_counts = {}
    total_count = 0  # To track total images across all classes

    for class_name in sorted(os.listdir(directory)):  # Sorting for better readability
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = len(os.listdir(class_path))
            image_counts[class_name] = count
            total_count += count

    return image_counts, total_count

# ✅ Count images in train and test sets
train_counts, total_train = count_images(train_dir)
test_counts, total_test = count_images(test_dir)

# ✅ Display image counts
print("📂 Image Counts in Training Directory:")
for class_name, count in train_counts.items():
    print(f"  {class_name}: {count} images")
print(f"🔹 Total Training Images: {total_train}")

print("\n📂 Image Counts in Test Directory:")
for class_name, count in test_counts.items():
    print(f"  {class_name}: {count} images")
print(f"🔹 Total Testing Images: {total_test}")

The dataset was randomly divided into training (80%) and testing (20%) sets before model training.

## Data Visualisation

Random images from both training and testing datasets are displayed to verify dataset organization and class diversity.

In [ ]:
def display_images_from_directory(directory, num_images=9):
    """Displays random images from a given directory containing class subfolders."""
    
    plt.figure(figsize=(12, 12))
    all_images = []

    # Collect all image paths from class folders
    for class_name in sorted(os.listdir(directory)):  # Sorted for consistency
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            images = [os.path.join(class_path, img) for img in os.listdir(class_path)]
            all_images.extend(images)

    if not all_images:
        print(f"⚠️ No images found in {directory}. Check the dataset path.")
        return
    
    num_images = min(num_images, len(all_images))  # Ensure we don't exceed available images

    # Select random images
    random_images = random.sample(all_images, num_images)
    
    for i, image_path in enumerate(random_images):
        image = cv2.imread(image_path)
        
        if image is None:
            print(f"⚠️ Unable to load image at {image_path}. Skipping.")
            continue
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        plt.subplot(3, 3, i + 1)
        plt.imshow(image)
        plt.axis("off")
    
    plt.show()

# ✅ Display random images from train & test sets
print("📌 Random Images from Training Set:")
display_images_from_directory(train_dir, num_images=9)

print("\n📌 Random Images from Test Set:")
display_images_from_directory(test_dir, num_images=9)

## CNN-LSTM Model

In [ ]:
IMG_SIZE = (128, 128)
def build_cnn_lstm_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)):
    model = Sequential()

    # CNN layers for feature extraction
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Dropout(0.3))

    # Global Average Pooling instead of Flatten to reduce spatial dimensions and keep important features
    model.add(GlobalAveragePooling2D())

    # Reshape the output to be compatible with LSTM input
    model.add(Reshape((1, model.output_shape[1])))  # Reshaping to (1, features)

    # LSTM layer to capture temporal dependencies (if your data is sequential, adjust accordingly)
    model.add(LSTM(128, activation='relu', return_sequences=False))

    # Dense layer for classification
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    
    # Output layer for classification (softmax for multi-class classification)
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model

## Model Training

In [ ]:
BATCH_SIZE = 32
# Define data generators for training and testing
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

# Flow data from directories in batches
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

In [ ]:
NUM_CLASSES = 39
# Train the CNN-LSTM model
cnn_lstm_model = build_cnn_lstm_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
cnn_lstm_model.summary()
cnn_lstm_model.fit(train_generator, epochs=10, validation_data=test_generator)

K.clear_session()

In [ ]:
cnn_lstm_accuracy = cnn_lstm_model.evaluate(test_generator)
print(f"CNN-LSTM Model Test Accuracy: {cnn_lstm_accuracy[1]}")

## Stable Diffusion Augmentation

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image
import os

# Load the pretrained model
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
train_classes = sorted(os.listdir(train_dir))
class_prompts = {cls: f"a leaf with {cls.replace('___', ' ').lower()}" for cls in train_classes}

In [ ]:
import signal

class TimeoutException(Exception):
    pass

def handler(signum, frame):
    raise TimeoutException("Image generation timed out.")

signal.signal(signal.SIGALRM, handler)

def generate_images_for_class(class_name, prompt, num_images=100, save_dir="generated_data"):
    output_dir = os.path.join(save_dir, class_name)
    os.makedirs(output_dir, exist_ok=True)

    for i in range(num_images):
        image_path = os.path.join(output_dir, f"{class_name}_{i}.png")
        if os.path.exists(image_path):  # Skip if already generated
            continue

        try:
            signal.alarm(60)  # timeout for one image
            image = pipe(prompt).images[0]
            image.save(image_path)
            signal.alarm(0)  # reset timer

            if i % 10 == 0:
                print(f"[{class_name}] Generated {i+1}/{num_images} images")

        except TimeoutException as te:
            print(f"[{class_name}] Image {i} timed out: {te}")
        except Exception as e:
            print(f"[{class_name}] Error at image {i}: {e}")

for cls, prompt in class_prompts.items():
    print(f"Generating for class: {cls}")
    generate_images_for_class(cls, prompt, num_images=50, save_dir="/kaggle/working/ldm_augmented")

## Retraining

In [ ]:
import shutil

augmented_dir = "/kaggle/working/ldm_augmented"

for cls in os.listdir(augmented_dir):
    src = os.path.join(augmented_dir, cls)
    dest = os.path.join(train_dir, cls)

    for img_file in os.listdir(src):
        shutil.copy(os.path.join(src, img_file), os.path.join(dest, img_file))

In [5]:
cnn_lstm_model = build_cnn_lstm_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
cnn_lstm_model.fit(train_generator, epochs=10, validation_data=test_generator)

NameError: name 'build_cnn_lstm_model' is not defined

In [ ]:
cnn_lstm_model.save('model_h5.keras')

## VGG16-LSTM

In [ ]:
from tensorflow.keras.models import load_model

# Load the previously trained CNN-LSTM model
model = load_model("model_h5.keras")

In [ ]:
def build_vgg16_lstm_model(input_shape=(128, 128, 3), num_classes=39):
    base_model = VGG16(input_shape=input_shape, include_top=False, weights='imagenet')

    for layer in base_model.layers:
        layer.trainable = False

    model = Sequential()
    model.add(base_model)
    model.add(GlobalAveragePooling2D())
    model.add(Reshape((1, -1)))  # Reshape for LSTM input
    model.add(LSTM(128, activation='relu', return_sequences=False))
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

    return model

In [ ]:
from tensorflow.keras.applications import VGG16

model_vgg_lstm = build_vgg16_lstm_model(input_shape=(128, 128, 3), num_classes=39)
model_vgg_lstm.summary()

# Train using the same generators
model_vgg_lstm.fit(train_generator, epochs=10, validation_data=test_generator)

# Evaluate
acc = model_vgg_lstm.evaluate(test_generator)
print(f"VGG16-LSTM Model Accuracy: {acc[1]}")

# Save if needed
model_vgg_lstm.save("vgg16_lstm_model.keras")

## Fine Tuning

In [ ]:
from tensorflow.keras.models import load_model

# Load the model before fine-tuning
model_before_fine_tune = load_model('vgg16_lstm_model.keras', compile=False)

In [ ]:
for layer in model.layers[:-2]:  # Freeze all but last 2 layers
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Fine-tune the model
history_fine_tune = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=10
)

In [ ]:
for layer in model.layers[-8:]:  # Unfreeze last 8 layers for fine-tuning
    layer.trainable = True

model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

fine_tune_now = model.fit(train_generator,
    validation_data=test_generator,
    epochs=10)

In [ ]:
for layer in model.layers:
    layer.trainable = True  # Unfreeze everything

model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

fine_tune_final = model.fit(train_generator,
    validation_data=test_generator,
    epochs=5)

In [ ]:
fine_tuned_test_loss, fine_tuned_test_accuracy = model.evaluate(test_generator)
print(f"Fine-Tuned Model Test Accuracy: {fine_tuned_test_accuracy:.2f}")

## Model Evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import os

def visualize_predictions(model, test_generator, class_indices, num_images=9):
    # Invert class_indices dict to get class names from indices
    idx_to_class = {v: k for k, v in class_indices.items()}
    
    # Get a batch of test images and labels
    x_test, y_test = next(test_generator)
    predictions = model.predict(x_test)
    
    # Choose random indices to visualize
    indices = random.sample(range(len(x_test)), num_images)

    plt.figure(figsize=(12, 12))
    for i, idx in enumerate(indices):
        image = x_test[idx]
        true_label = idx_to_class[np.argmax(y_test[idx])]
        predicted_label = idx_to_class[np.argmax(predictions[idx])]
        
        plt.subplot(3, 3, i + 1)
        plt.imshow(image)
        plt.title(f"True: {true_label}\nPred: {predicted_label}", color="green" if true_label == predicted_label else "red")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# 🔍 Example usage:
visualize_predictions(model_vgg_lstm, test_generator, test_generator.class_indices, num_images=9)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def evaluate_model(model, test_generator):
    # Predict the entire test set
    Y_pred = model.predict(test_generator)
    y_pred = np.argmax(Y_pred, axis=1)

    # Get ground truth labels
    y_true = test_generator.classes
    class_labels = list(test_generator.class_indices.keys())

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot Confusion Matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

# 🔍 Example usage:
evaluate_model(model_vgg_lstm, test_generator)  # or cnn_lstm_model

In [ ]:
model.save('finetuned.keras')

## Saliency Maps

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Load model
model = tf.keras.models.load_model("finetuned.keras")

# Get a batch of test images
batch = next(iter(test_generator))
if isinstance(batch, tuple):
    test_images = batch[0]  # Get all images from the batch
    # If labels are available and you want to check them
    test_labels = batch[1]
    print(f"Available images in batch: {test_images.shape[0]}")
    # You can print labels if needed
    if hasattr(test_generator, 'class_indices'):
        label_map = {v: k for k, v in test_generator.class_indices.items()}
        if len(test_labels.shape) > 1:  # One-hot encoded labels
            label_indices = np.argmax(test_labels, axis=1)
            print("Image labels:", [label_map.get(idx, idx) for idx in label_indices[:5]], "...")
        else:  # Integer labels
            print("Image labels:", [label_map.get(idx, idx) for idx in test_labels[:5]], "...")
else:
    test_images = batch
    print(f"Available images in batch: {test_images.shape[0]}")

# Select a different image (change index as needed)
image_index = 3  # Try different indices: 0, 1, 2, 3, etc.
if image_index < test_images.shape[0]:
    test_image = test_images[image_index:image_index+1]
    print(f"Selected image at index {image_index}")
else:
    print(f"Index {image_index} out of range, defaulting to first image")
    test_image = test_images[0:1]

# Print shape
print("Test image shape:", test_image.shape)

# Check if test_image is already a numpy array
if isinstance(test_image, np.ndarray):
    # Convert to TensorFlow tensor if needed for gradient calculations
    test_image_tensor = tf.convert_to_tensor(test_image)
else:
    test_image_tensor = test_image
    # If it's a tensor, we might need the numpy version later
    test_image = test_image.numpy() if hasattr(test_image, 'numpy') else np.array(test_image)

# Print model structure to understand it better
print("Model layers:")
for i, layer in enumerate(model.layers):
    print(f"{i}: {layer.name}, type: {type(layer).__name__}")
    
    # If it's a sequential or functional model, print its layers too
    if hasattr(layer, 'layers'):
        for j, nested_layer in enumerate(layer.layers):
            print(f"  {i}.{j}: {nested_layer.name}, type: {type(nested_layer).__name__}")

# First, run inference to make sure the model is built
predictions = model.predict(test_image_tensor)
print("Predicted class index:", np.argmax(predictions[0]))

# Find the last convolutional layer by rebuilding submodels
# The error suggests there's a nested sequential_1 model
sequential_model = None
for layer in model.layers:
    if layer.name == 'sequential_1':
        sequential_model = layer
        break

if sequential_model is None:
    print("No 'sequential_1' layer found. Let's try another approach.")
    
    # Try direct visualization without creating a grad model
    # This is a simpler approach that doesn't require building a new model
    with tf.GradientTape() as tape:
        # Watch the input image
        tape.watch(test_image_tensor)
        # Get the prediction
        prediction = model(test_image_tensor)
        # Get index of target class
        target_class_idx = tf.argmax(prediction[0])
        # Get the target class output
        target_class_output = prediction[:, target_class_idx]
    
    # Get the gradients of the target class with respect to the input image
    gradients = tape.gradient(target_class_output, test_image_tensor)
    
    # Get the mean of absolute gradients for each pixel to create a saliency map
    saliency_map = tf.reduce_mean(tf.abs(gradients), axis=-1)
    
    # Normalize the saliency map
    saliency_map = (saliency_map - tf.reduce_min(saliency_map)) / (
        tf.reduce_max(saliency_map) - tf.reduce_min(saliency_map) + tf.keras.backend.epsilon()
    )
    
    # Convert to numpy for visualization
    saliency_map = saliency_map.numpy()[0]
    
    # Create visualization
    plt.figure(figsize=(10, 8))
    plt.subplot(1, 2, 1)
    plt.imshow(test_image[0])
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    plt.imshow(saliency_map, cmap='jet')
    plt.title('Saliency Map')
    plt.axis('off')
    
    # Superimpose the saliency map on the original image
    plt.figure(figsize=(8, 8))
    saliency_map_resized = tf.image.resize(
        tf.expand_dims(tf.expand_dims(saliency_map, -1), 0),
        (test_image.shape[1], test_image.shape[2])
    ).numpy()[0, :, :, 0]
    
    # Convert to RGB colormap
    saliency_colormap = cm.jet(saliency_map_resized)[:, :, :3]
    
    # Create superimposed image
    superimposed = test_image[0] * 0.6 + saliency_colormap * 0.4
    
    # Ensure values are in valid range
    superimposed = superimposed / np.max(superimposed)
    
    plt.imshow(superimposed)
    plt.title('Saliency Overlay')
    plt.axis('off')
    plt.show()
    
else:
    # If sequential_1 exists, we need to modify our approach
    print("Found sequential_1 submodel, examining its layers...")
    
    # Find the last conv layer in the sequential model
    last_conv_layer = None
    for layer in reversed(sequential_model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_layer = layer
            print(f"Found last convolutional layer: {layer.name}")
            break
    
    if last_conv_layer is None:
        print("No convolutional layer found in sequential_1. Let's try a broader search...")
        
        # Try to find any Conv2D layer in the model
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_layer = layer
                print(f"Found convolutional layer in main model: {layer.name}")
                break
            elif hasattr(layer, 'layers'):
                for nested_layer in layer.layers:
                    if isinstance(nested_layer, tf.keras.layers.Conv2D):
                        last_conv_layer = nested_layer
                        print(f"Found nested convolutional layer: {nested_layer.name}")
                        break
                if last_conv_layer is not None:
                    break
    
    if last_conv_layer is None:
        print("No convolutional layers found in the model. Using the saliency map approach...")
        
        # Use the same saliency map approach as above
        with tf.GradientTape() as tape:
            tape.watch(test_image_tensor)
            prediction = model(test_image_tensor)
            target_class_idx = tf.argmax(prediction[0])
            target_class_output = prediction[:, target_class_idx]
        
        gradients = tape.gradient(target_class_output, test_image_tensor)
        saliency_map = tf.reduce_mean(tf.abs(gradients), axis=-1)
        saliency_map = (saliency_map - tf.reduce_min(saliency_map)) / (
            tf.reduce_max(saliency_map) - tf.reduce_min(saliency_map) + tf.keras.backend.epsilon()
        )
        
        saliency_map = saliency_map.numpy()[0]
        
        # Visualization code (same as above)
        plt.figure(figsize=(10, 8))
        plt.subplot(1, 2, 1)
        plt.imshow(test_image[0])
        plt.title('Original Image')
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.imshow(saliency_map, cmap='jet')
        plt.title('Saliency Map')
        plt.axis('off')
        
        plt.figure(figsize=(8, 8))
        saliency_map_resized = tf.image.resize(
            tf.expand_dims(tf.expand_dims(saliency_map, -1), 0),
            (test_image.shape[1], test_image.shape[2])
        ).numpy()[0, :, :, 0]
        
        saliency_colormap = cm.jet(saliency_map_resized)[:, :, :3]
        superimposed = test_image[0] * 0.6 + saliency_colormap * 0.4
        superimposed = superimposed / np.max(superimposed)
        
        plt.imshow(superimposed)
        plt.title('Saliency Overlay')
        plt.axis('off')
        plt.show()
        
    else:
        # We found a convolutional layer, so let's try the Feature Map approach
        # This avoids the problem with model.get_layer() by using a different technique
        
        print("Using feature map visualization approach...")
        
        # Run the model to get predictions
        with tf.GradientTape() as tape:
            # Forward pass to extract activations
            activations = []
            x = test_image_tensor
            
            # Manual forward pass through the model to capture activations
            for layer in model.layers:
                if isinstance(layer, tf.keras.layers.Conv2D):
                    x = layer(x)
                    activations.append(x)
                elif hasattr(layer, 'layers'):  # Sequential or Functional model
                    for nested_layer in layer.layers:
                        if isinstance(nested_layer, tf.keras.layers.Conv2D):
                            x = nested_layer(x)
                            activations.append(x)
                        else:
                            x = nested_layer(x)
                else:
                    x = layer(x)
            
            predictions = x  # Final output
            target_class_idx = tf.argmax(predictions[0])
            print(f"Predicted class: {target_class_idx.numpy()}")
            target_class_output = predictions[:, target_class_idx]
        
        if not activations:
            print("No activations collected. Using saliency map approach...")
            # Use saliency map approach (same code as above)
            gradients = tape.gradient(target_class_output, test_image_tensor)
            saliency_map = tf.reduce_mean(tf.abs(gradients), axis=-1)
            saliency_map = (saliency_map - tf.reduce_min(saliency_map)) / (
                tf.reduce_max(saliency_map) - tf.reduce_min(saliency_map) + tf.keras.backend.epsilon()
            )
            saliency_map = saliency_map.numpy()[0]
            
            # Visualization code (same as above)
            plt.figure(figsize=(10, 8))
            plt.subplot(1, 2, 1)
            plt.imshow(test_image[0])
            plt.title('Original Image')
            plt.axis('off')
            
            plt.subplot(1, 2, 2)
            plt.imshow(saliency_map, cmap='jet')
            plt.title('Saliency Map')
            plt.axis('off')
            
            plt.figure(figsize=(8, 8))
            saliency_map_resized = tf.image.resize(
                tf.expand_dims(tf.expand_dims(saliency_map, -1), 0),
                (test_image.shape[1], test_image.shape[2])
            ).numpy()[0, :, :, 0]
            
            saliency_colormap = cm.jet(saliency_map_resized)[:, :, :3]
            superimposed = test_image[0] * 0.6 + saliency_colormap * 0.4
            superimposed = superimposed / np.max(superimposed)
            
            plt.imshow(superimposed)
            plt.title('Saliency Overlay')
            plt.axis('off')
            plt.show()
        else:
            # We have activations, so use the last conv layer's activations for visualization
            last_conv_activation = activations[-1]
            
            # Calculate gradients
            grads = tape.gradient(target_class_output, last_conv_activation)
            
            # Pool gradients
            pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
            
            # Multiply each feature map with its importance
            last_conv_activation = last_conv_activation[0]
            heatmap = last_conv_activation @ pooled_grads[..., tf.newaxis]
            heatmap = tf.squeeze(heatmap)
            
            # Normalize between 0 & 1
            heatmap = tf.maximum(heatmap, 0) / tf.maximum(tf.reduce_max(heatmap), 1e-10)
            heatmap = heatmap.numpy()
            
            # Display original image
            plt.figure(figsize=(10, 8))
            plt.subplot(1, 2, 1)
            plt.imshow(test_image[0])
            plt.title('Original Image')
            plt.axis('off')
            
            # Display heatmap
            plt.subplot(1, 2, 2)
            plt.imshow(heatmap, cmap='jet')
            plt.title('Activation Heatmap')
            plt.axis('off')
            
            # Optional: Superimpose heatmap on original image
            plt.figure(figsize=(8, 8))
            img = test_image[0]
            heatmap_resized = tf.image.resize(
                tf.expand_dims(heatmap, -1), 
                (img.shape[0], img.shape[1])
            ).numpy()
            
            # Convert heatmap to RGB
            heatmap_jet = cm.jet(heatmap_resized.squeeze())[:, :, :3]
            
            # Create superimposed image (0.4 transparency for heatmap)
            superimposed = img * 0.6 + heatmap_jet * 0.4
            
            # Ensure values are in valid range
            superimposed = superimposed / np.max(superimposed)
            
            plt.imshow(superimposed)
            plt.title('Heatmap Overlay')
            plt.axis('off')
            plt.show()

## LIME Explainability

In [ ]:
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np

# LIME Explanation
explainer = lime_image.LimeImageExplainer()

def model_predict(imgs):
    # Make sure images are in correct format
    # LIME expects batch of images
    preds = model.predict(imgs)
    return preds

# Explain the instance
explanation = explainer.explain_instance(
    test_image[0],  # Use the selected image
    model_predict,  # Prediction function
    top_labels=3,   # Get explanation for top 3 classes
    hide_color=0,   # Black is the hidden color
    num_samples=1000, # More samples = better explanation but slower
    segmentation_fn=None # Use quickshift segmentation
)

# Get predicted class
top_label = explanation.top_labels[0]

# Plot multiple explanations with different settings
plt.figure(figsize=(20, 10))

# Original image
plt.subplot(2, 3, 1)
plt.imshow(test_image[0])
plt.title('Original Image')
plt.axis('off')

# Top positive features only
plt.subplot(2, 3, 2)
temp, mask = explanation.get_image_and_mask(
    top_label, 
    positive_only=True, 
    num_features=5, 
    hide_rest=False
)
plt.imshow(mark_boundaries(temp, mask, color=(1, 1, 0), outline_color=(1, 1, 0)))
plt.title('Top 5 Positive Features')
plt.axis('off')

# All positive and negative features
plt.subplot(2, 3, 3)
temp, mask = explanation.get_image_and_mask(
    top_label, 
    positive_only=False, 
    negative_only=False,
    num_features=10, 
    hide_rest=False
)
plt.imshow(mark_boundaries(temp, mask))
plt.title('Top 10 Features (Positive & Negative)')
plt.axis('off')

# Hide the non-important parts
plt.subplot(2, 3, 4)
temp, mask = explanation.get_image_and_mask(
    top_label, 
    positive_only=True, 
    num_features=5, 
    hide_rest=True
)
plt.imshow(mark_boundaries(temp, mask))
plt.title('Only Top 5 Features (Rest Hidden)')
plt.axis('off')

# Get the explanation as a heatmap
plt.subplot(2, 3, 5)
# Get the map of feature importance
ind = explanation.top_labels[0]
exp_data = explanation.local_exp[ind]
exp_data = sorted(exp_data, key=lambda x: x[0])
temp_img = np.zeros(explanation.segments.shape)
for i, (segment, value) in enumerate(exp_data):
    temp_img[explanation.segments == segment] = value

# Normalize for visualization
temp_img = (temp_img - np.min(temp_img)) / (np.max(temp_img) - np.min(temp_img))
plt.imshow(temp_img, cmap='hot')
plt.colorbar()
plt.title('Feature Importance Heatmap')
plt.axis('off')

# Show the probability distribution from the model
plt.subplot(2, 3, 6)
# Get model predictions
preds = model.predict(np.expand_dims(test_image[0], axis=0))[0]
# Get class names if available
class_names = None
if hasattr(test_generator, 'class_indices'):
    class_names = {v: k for k, v in test_generator.class_indices.items()}

# Plot top 5 classes
top_5_idx = np.argsort(preds)[-5:][::-1]
top_5_values = preds[top_5_idx]
if class_names:
    labels = [class_names.get(idx, f"Class {idx}") for idx in top_5_idx]
else:
    labels = [f"Class {idx}" for idx in top_5_idx]

# Create horizontal bar plot
plt.barh(range(len(top_5_idx)), top_5_values)
plt.yticks(range(len(top_5_idx)), labels)
plt.xlabel('Probability')
plt.title('Top 5 Predictions')

plt.tight_layout()
plt.show()

# Print the model's confidence for the top class
top_class_idx = np.argmax(preds)
top_class_name = class_names.get(top_class_idx, f"Class {top_class_idx}") if class_names else f"Class {top_class_idx}"
print(f"The model predicts '{top_class_name}' with {preds[top_class_idx]*100:.2f}% confidence")

# Analysis of which features contributed most
print("\nFeature importance analysis:")
for segment, value in sorted(exp_data, key=lambda x: abs(x[1]), reverse=True)[:5]:
    feature_type = "positive" if value > 0 else "negative"
    percentage = abs(value) / sum(abs(val) for _, val in exp_data) * 100
    print(f"Segment {segment}: {feature_type} contribution of {value:.4f} ({percentage:.1f}% of total importance)")